<a href="https://colab.research.google.com/github/Davidson-Open-University/csc203-labs/blob/main/notebooks/CSC203-Lab1-Sets-in-Python.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 203 Lab 1: Sets in Python

**Study Session 1: Set Theory** · Davidson Open University Global

**In this lab you will:**
- create sets in Python and test membership and subsets;
- compute union, intersection, difference, complement and set-builder sets;
- build the power set and the Cartesian product, and check |P(A)| = 2ⁿ;
- represent a relation as a set of ordered pairs.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the examples and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short Python function and test it.
- **Part D – Apply and reflect** (about 10 minutes): use your tools on your own example and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `csc203-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

*This lab is practice to help you learn and prepare for the assignment. It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), 6) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k) + '=' + _norm(v) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), 6) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('csc203|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False)
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '1.1': {'h': '269463f033ae648d', 'hint': 'Union uses the | symbol: everything in A or in B.', 'ok': 'The union contains every element of A and every element of B.'},
    '1.2': {'h': '7647908816f83442', 'hint': 'Intersection uses &. Which elements are in both A and D? (The answer may be the empty set, set().)', 'ok': 'A and D have no elements in common, so A ∩ D = ∅: they are disjoint.'},
    '1.3': {'h': '7d0e60974910634d', 'hint': 'The complement is "everything in U that is not in A": U - A.'},
    '1.4': {'h': 'c7ce01e17a72a8b1', 'hint': 'Work out Dᶜ = U - D and (A ∩ B)ᶜ = U - (A & B) first, then combine with & and |. Use brackets.', 'ok': 'This is SAQ 2a(ii) from Study Session 1.'},
    '1.5': {'h': '1657758aad53cf75', 'hint': 'Copy the pattern {x for x in U if ...}. A multiple of 3 has remainder 0: x % 3 == 0.'},
    '1.6': {'h': 'c0ccfc9cc96518fd', 'hint': '|A ∩ B| = |A| + |B| − |A ∪ B|.', 'ok': 'Two students take both subjects.'},
    '1.7': {'h': '63d81bcb2e0118db', 'hint': 'You should get 2ⁿ subsets, including set() (the empty set) and S itself. Check both loops.', 'ok': 'Your function builds the power set.', 'args': [({1, 2, 3},), (set(),), ({'x', 'y'},)]},
    '1.8': {'h': 'b9307a5094b5d50b', 'hint': 'Use two for-clauses in one set comprehension. Order matters inside the pair: (a, b), with a from A.', 'ok': 'Notice that |A × B| = |A| × |B|.', 'args': [({1, 2}, {'x', 'y', 'z'}), ({1, 2, 3}, {4, 5, 6, 7}), (set(), {1})]},
    '1.9': {'h': '23e58d14d8b7a8c6', 'hint': 'Start from cartesian(S, S) and keep the pairs where b % a == 0. Remember (a, a) is included because every number divides itself.', 'ok': 'You will meet this "divides" relation again in Lab 4.'},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
Python has a built-in **set** type that behaves like the sets in Study Session 1:
- a set has **no repeated elements**, and its elements have **no order**;
- you write a set with curly brackets, for example `{1, 2, 3}`;
- `x in A` tests **membership** (x ∈ A).

**Predict first:** what will Python print for the set `{3, 1, 3, 2, 1}`? Write your guess down, then run the cell.

In [ ]:
A = {3, 1, 3, 2, 1}
print('A =', A)                 # repeats disappear
print('Number of elements |A| =', len(A))
print('Is 2 in A?', 2 in A)     # 2 ∈ A
print('Is 5 in A?', 5 in A)     # 5 ∉ A

Python uses these symbols for the set operations from the study session:

| Maths | Python | Meaning |
|---|---|---|
| A ∪ B | `A \| B` | union: in A **or** B (or both) |
| A ∩ B | `A & B` | intersection: in **both** A and B |
| A \ B | `A - B` | difference: in A but **not** in B |
| B ⊆ A | `B <= A` | B is a subset of A |
| Aᶜ | `U - A` | complement: everything in the universal set U that is not in A |

Run the cell and check each result by hand.

In [ ]:
U = set(range(1, 11))     # the universal set {1, 2, ..., 10}
A = {1, 2, 3, 4}
B = {3, 4, 5}
print('A ∪ B =', A | B)
print('A ∩ B =', A & B)
print('A \\ B =', A - B)
print('B \\ A =', B - A)
print('Aᶜ    =', U - A)
print('Is {3, 4} ⊆ A?', {3, 4} <= A)

**Set-builder notation in Python.** The set {x ∈ U : x is odd} is written as a *set comprehension*:
`{x for x in U if x % 2 == 1}`. Read it as "the set of x, for each x in U, if x is odd". (`x % 2` is the remainder when x is divided by 2.)

In [ ]:
odds = {x for x in U if x % 2 == 1}
squares = {x * x for x in U if x <= 4}
print('Odd numbers in U:', odds)
print('Squares of 1 to 4:', squares)

## Part B – Practise (about 20 minutes)
For every question, replace `...` with Python code, run the cell, then run the **check** cell under it.

Use these sets (run the cell first):

In [ ]:
U = set(range(1, 11))
A = {1, 2, 3, 6, 7}
B = {2, 3, 4}
D = {4, 5, 8, 9}
print('U =', U); print('A =', A); print('B =', B); print('D =', D)

In [ ]:
answer_1_1 = ...   # A ∪ B

In [ ]:
check('1.1', answer_1_1)

In [ ]:
answer_1_2 = ...   # A ∩ D

In [ ]:
check('1.2', answer_1_2)

In [ ]:
answer_1_3 = ...   # the complement of A, Aᶜ (inside U)

In [ ]:
check('1.3', answer_1_3)

In [ ]:
answer_1_4 = ...   # (A ∩ Dᶜ) ∪ (A ∩ B)ᶜ  — build it one piece at a time

In [ ]:
check('1.4', answer_1_4)

In [ ]:
answer_1_5 = ...   # {x ∈ U : x is a multiple of 3}, using a set comprehension

In [ ]:
check('1.5', answer_1_5)

**Counting with sets.** The rule |A ∪ B| = |A| + |B| − |A ∩ B| lets you count a union without listing it.

In a class of students, |A| = 5 take Mathematics, |B| = 8 take Physics and |A ∪ B| = 11 take at least one of them. How many take **both**? (This is SAQ 7.) Rearrange the rule, then let Python do the arithmetic.

In [ ]:
answer_1_6 = ...   # |A ∩ B| when |A| = 5, |B| = 8 and |A ∪ B| = 11

In [ ]:
check('1.6', answer_1_6)

## Part C – Build a tool (about 20 minutes)
### C1. The power set
The **power set** P(S) is the set of *all* subsets of S, including ∅ and S itself. If |S| = n, then |P(S)| = 2ⁿ.

`combinations(items, r)` from `itertools` gives every way to choose r items. Complete the function so that it returns a **list of sets**: all the subsets of size 0, 1, 2, …, len(S).

*Hint:* loop `for r in range(len(items) + 1):` and inside it `for c in combinations(items, r):`, adding `set(c)` to the list.

In [ ]:
from itertools import combinations

def power_set(S):
    items = list(S)
    subsets = ...    # YOUR CODE: start with an empty list [], then use two loops to add every subset (as a set)
    return subsets

print(power_set({'a', 'b', 'c'}))

In [ ]:
check('1.7', lambda S: sorted(map(sorted, power_set(S))) if power_set(S) is not ... else None)

In [ ]:
# Check the rule |P(S)| = 2ⁿ for sets of size 0 to 6
for n in range(7):
    S = set(range(n))
    ps = power_set(S)
    print(f'|S| = {n}:  |P(S)| = {len(ps) if isinstance(ps, list) else "?"},  2**{n} = {2 ** n}')

### C2. The Cartesian product and relations
The **Cartesian product** A × B is the set of all ordered pairs (a, b) with a ∈ A and b ∈ B. Python writes an ordered pair as a tuple `(a, b)`.
Complete the function with a set comprehension: `{(a, b) for a in A for b in B}`.

In [ ]:
def cartesian(A, B):
    # YOUR CODE: return the set of all pairs (a, b)
    ...

print(cartesian({1, 2}, {'x', 'y', 'z'}))

In [ ]:
check('1.8', cartesian)

A **relation** R from A to B is any *subset* of A × B. For example, on S = {1, 2, 3, 4, 6, 12} the relation "a divides b" is
R = {(a, b) ∈ S × S : b % a == 0}.

Build this relation using `cartesian` and a set comprehension with a condition.

In [ ]:
S = {1, 2, 3, 4, 6, 12}
answer_1_9 = ...   # the set of pairs (a, b) in S × S where a divides b
print(answer_1_9)

In [ ]:
check('1.9', answer_1_9)

## Part D – Apply and reflect (about 10 minutes)
**D1. Your own data (prepares you for Assignment 1).** In the cell below, make a small universal set U from your own life, for example the courses offered in your department, and two sets A and B, for example the courses taken by two students. Print A ∪ B, A ∩ B, A \ B and the complement of A. There is no automatic check here: check your results by hand.

In [ ]:
# D1: your own example
U = {...}
A = {...}
B = {...}
# print the union, intersection, difference and complement here

**D2. Reflect.** Answer in one or two sentences each:
1. Python printed `{1, 2, 3}` when you typed `{3, 1, 3, 2, 1}`. Which two properties of sets does this show?
2. Why is (1, 2) ≠ (2, 1) as ordered pairs, but {1, 2} = {2, 1} as sets?
3. How many subsets does a set with 10 elements have? Why would listing them by hand be a bad idea?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `csc203-labs` repository → commit message *"Lab 1 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.